## WebScraping using ScrapegraphAI



 Scrapegraph uses AI to simplify web scraping. Instead of writing complex code, you tell it what data you want, and it figures out how to extract it. It works on websites and even local files like HTML.



In [1]:
%%capture
!pip install scrapegraphai
!apt install chromium-chromedriver
!pip install nest-asyncio
!pip install playwright
!playwright install

In [19]:
!playwright install-deps


In [20]:
!pip install langchain-google-genai

  Using cached filetype-1.2.0-py2.py3-none-any.whl.metadata (6.5 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyasn1-0.6.3-py3-none-any.whl.metadata (8.4 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
Using cached filetype-1.2.0-py2.py3-none-any.whl (19 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.5/832.5 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 9.2 MB/s eta 0:00:0000:01:00:01
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)
Using cached pyasn1-0.6.3-py3-none-any.whl (83 kB)
Using cached pycparser-3.0-py3-none-any.whl (48 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [3]:
import nest_asyncio
nest_asyncio.apply()

### Scraping Websites



In [ ]:
OPENAI_API_KEY = ""  # Set your OpenAI API key here (optional). If you have one, paste it between the quotes.
GOOGLE_API_KEY  = ""  # Existing Google API key (used for Google/Gemini)


In [5]:
# Define LLM configurations: prefer OpenAI if available, otherwise explicit Google provider for Gemini
graph_config_openai = {
    "llm": {
        "api_key": OPENAI_API_KEY,
        "model": "gpt-4o",
        # optionally add "provider": "openai" if needed by your environment
    },
    "verbose": True,
    "headless": True,
}

graph_config_google = {
    "llm": {
        "api_key": GOOGLE_API_KEY,
        "model": "google_genai/gemini-2.5-flash",  # Prefixed with the correct LangChain provider identifier
        "model_tokens": 8192,
        "format": "json",
    },
    "embeddings": {
        "api_key": GOOGLE_API_KEY,
        "model": "google_genai/text-embedding-004", # Prefixed here as well
    },
    "verbose": True,
    "headless": True,
}

graph_config_ollama = {
    "llm": {
        "model": "ollama/llama3:8b",
        "base_url": "http://localhost:11434",  # default Ollama server URL
    },
    "embeddings": {
        "model": "ollama/nomic-embed-text",    # embeddings are required!
        "base_url": "http://localhost:11434",
    },
    "verbose": True,
    "headless": True,
}

In [31]:
import json
from scrapegraphai.graphs import SmartScraperGraph

# Choose configuration: prefer OpenAI if API key provided, otherwise use Google/Gemini config
selected_config = graph_config_google if GOOGLE_API_KEY else graph_config_ollama


# Create the SmartScraperGraph instance
smart_scraper_graph = SmartScraperGraph(
    prompt="Find some information about what does the company do, the name and a contact email.",
    source="https://www.buildfastwithai.com",
    config=selected_config
)

# Run the pipeline with basic error handling for unsupported providers
try:
    result = smart_scraper_graph.run()
    print(json.dumps(result, indent=4))
except ValueError as e:
    print("ValueError:", e)
    print("Hint: ensure the selected config.llm.provider is supported by your scrapegraphai version, or provide an OpenAI key and use graph_config_openai.")

Unexpected argument 'model_tokens' provided to ChatGoogleGenerativeAI. Did you mean: 'max_tokens'?
Unexpected argument 'format' provided to ChatGoogleGenerativeAI.
--- Executing Fetch Node ---
--- (Fetching HTML from: https://www.buildfastwithai.com) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
Processing chunks: 0it [00:00, ?it/s]
{'content': "I apologize, but the provided 'WEBSITE CONTENT' is empty. Therefore, I cannot find any information about what the company does, its name, or a contact email."}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨


{
    "content": "I apologize, but the provided 'WEBSITE CONTENT' is empty. Therefore, I cannot find any information about what the company does, its name, or a contact email."
}


In [7]:
import json
from scrapegraphai.graphs import SmartScraperGraph


smart_scraper_graph = SmartScraperGraph(
    prompt="Fetch the prices of all the latest LLMs",
    # also accepts a string with the already downloaded HTML code
    source="https://groq.com/pricing/",
    config=graph_config_ollama
)

result = smart_scraper_graph.run()

print(json.dumps(result,indent=4))

--- Executing Fetch Node ---
--- (Fetching HTML from: https://groq.com/pricing/) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
{'content': "To fetch the prices of all the latest LLMs, I've extracted the relevant information from the scraped content. Here are the prices for each model:\r\n\r\n* GPT OSS 20B: $0.075 (input) and $0.30 (output) per million tokens.\r\n* GPT OSS Safeguard 20B: same as above.\r\n* GPT OSS 120B: $0.15 (input) and $0.60 (output) per million tokens.\r\n* Llama 4 Scout (17Bx16E): $0.11 (input) and $0.34 (output) per million tokens.\r\n* Qwen3 32B: $0.29 (input) and $0.59 (output) per million tokens.\r\n* Llama 3.3 70B Versatile: $0.59 (input) and $0.79 (output) per million tokens.\r\n* Llama 3.1 8B Instant: $0.05 (input) and $0.08 (output) per million tokens.\r\n\r\nNote that these prices are subject to change, and additional fees may apply for certain features or services."}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scra

{
    "content": "To fetch the prices of all the latest LLMs, I've extracted the relevant information from the scraped content. Here are the prices for each model:\r\n\r\n* GPT OSS 20B: $0.075 (input) and $0.30 (output) per million tokens.\r\n* GPT OSS Safeguard 20B: same as above.\r\n* GPT OSS 120B: $0.15 (input) and $0.60 (output) per million tokens.\r\n* Llama 4 Scout (17Bx16E): $0.11 (input) and $0.34 (output) per million tokens.\r\n* Qwen3 32B: $0.29 (input) and $0.59 (output) per million tokens.\r\n* Llama 3.3 70B Versatile: $0.59 (input) and $0.79 (output) per million tokens.\r\n* Llama 3.1 8B Instant: $0.05 (input) and $0.08 (output) per million tokens.\r\n\r\nNote that these prices are subject to change, and additional fees may apply for certain features or services."
}


In [11]:
smart_scraper_graph = SmartScraperGraph(
    prompt="Fetch all the rain pant products and their prices",
    # also accepts a string with the already downloaded HTML code
    source="https://www.decathlon.in/c/men-rainwear-13098?inStock=1",
    config=graph_config_ollama
)

result = smart_scraper_graph.run()

print(json.dumps(result,indent=2))

--- Executing Fetch Node ---
--- (Fetching HTML from: https://www.decathlon.in/c/men-rainwear-13098?inStock=1) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
{'content': [{'product': 'Quechua Men Rain Pant Compact Waterproof Hiking Over Trousers with Ankle Zip and Pockets - Black', 'price': '999'}, {'product': 'Quechua Raincoat Men Waterproof 1/2 Zip Jacket with 5000mm Coating - Black', 'price': '699'}, {'product': "Quechua Rain coat Men's Compact Lightweight Breathable Waterproof Rain Jacket and Pant Set - Black", 'price': '1599'}, {'product': 'Forclaz Adult Waterproof Poncho for 0-30L Bag Dark Blue - MT100', 'price': '1299'}, {'product': 'Caperlan Rain Waterproof Poncho Improved Quality with Storage Pouch - Black', 'price': '999'}, {'product': 'Quechua Adult Waterproof Poncho for 0-10L Bag Pale Blue - MT50', 'price': '799'}, {'product': 'Btwin Overtrousers Unisex City 100 Waterproof Urban Commuting Cycling Lightweight Packable Protection - Assorted', 'pric

{
  "content": [
    {
      "product": "Quechua Men Rain Pant Compact Waterproof Hiking Over Trousers with Ankle Zip and Pockets - Black",
      "price": "999"
    },
    {
      "product": "Quechua Raincoat Men Waterproof 1/2 Zip Jacket with 5000mm Coating - Black",
      "price": "699"
    },
    {
      "product": "Quechua Rain coat Men's Compact Lightweight Breathable Waterproof Rain Jacket and Pant Set - Black",
      "price": "1599"
    },
    {
      "product": "Forclaz Adult Waterproof Poncho for 0-30L Bag Dark Blue - MT100",
      "price": "1299"
    },
    {
      "product": "Caperlan Rain Waterproof Poncho Improved Quality with Storage Pouch - Black",
      "price": "999"
    },
    {
      "product": "Quechua Adult Waterproof Poncho for 0-10L Bag Pale Blue - MT50",
      "price": "799"
    },
    {
      "product": "Btwin Overtrousers Unisex City 100 Waterproof Urban Commuting Cycling Lightweight Packable Protection - Assorted",
      "price": "999"
    },
    {
      "pro

In [12]:
smart_scraper_graph = SmartScraperGraph(
    prompt="Fetch all the waterproof biking shoes products and their prices",
    # also accepts a string with the already downloaded HTML code
    source="https://www.flipkart.com/search?q=waterproof+bike+riding+shoes+for+men&sid=osp%2Ccil&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_1_28_sc_na_ps&otracker1=AS_QueryStore_OrganicAutoSuggest_1_28_sc_na_ps&as-pos=1&as-type=RECENT&suggestionId=waterproof+bike+riding+shoes+for+men%7CMen%27s+Footwear&requestId=9f15223d-ae32-4045-bcd4-19d44ac0248f&as-searchtext=riding%20shoes%20bike%20waterprrof",
    config=graph_config_ollama
)

result = smart_scraper_graph.run()

print(json.dumps(result,indent=2))

--- Executing Fetch Node ---
--- (Fetching HTML from: https://www.flipkart.com/search?q=waterproof+bike+riding+shoes+for+men&sid=osp%2Ccil&as=on&as-show=on&otracker=AS_QueryStore_OrganicAutoSuggest_1_28_sc_na_ps&otracker1=AS_QueryStore_OrganicAutoSuggest_1_28_sc_na_ps&as-pos=1&as-type=RECENT&suggestionId=waterproof+bike+riding+shoes+for+men%7CMen%27s+Footwear&requestId=9f15223d-ae32-4045-bcd4-19d44ac0248f&as-searchtext=riding%20shoes%20bike%20waterprrof) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
Processing chunks: 100%|██████████| 5/5 [00:00<00:00, 10968.37it/s]
{'content': [{'product': 'Shimano SH-WA05', 'price': 230.0}, {'product': 'Specialized Shoes, Veneto', 'price': 250.0}, {'product': 'Pearl Izumi PRO Insoles for Shimano Shoes', 'price': None}, {'product': 'Lake MX235', 'price': 180.0}, {'product': 'Shimano SH-WA05-BC', 'price': 220.0}]}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨


{
  "content": [
    {
      "product": "Shimano SH-WA05",
      "price": 230.0
    },
    {
      "product": "Specialized Shoes, Veneto",
      "price": 250.0
    },
    {
      "product": "Pearl Izumi PRO Insoles for Shimano Shoes",
      "price": null
    },
    {
      "product": "Lake MX235",
      "price": 180.0
    },
    {
      "product": "Shimano SH-WA05-BC",
      "price": 220.0
    }
  ]
}


In [9]:
smart_scraper_graph = SmartScraperGraph(
    prompt="Fetch all the products and their prices",
    # also accepts a string with the already downloaded HTML code
    source="https://orae.in",
    config=graph_config_ollama
)

result = smart_scraper_graph.run()

print(json.dumps(result,indent=2))

--- Executing Fetch Node ---
--- (Fetching HTML from: https://orae.in) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
{'content': [{'name': 'Sequence Cork Yoga Mat', 'price': '₹4,299.00'}, {'name': 'Loom Cotton Yoga Mat', 'price': '₹1,999.00'}, {'name': 'Pose Cork Yoga Mat', 'price': '₹4,799.00'}]}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨


{
  "content": [
    {
      "name": "Sequence Cork Yoga Mat",
      "price": "\u20b94,299.00"
    },
    {
      "name": "Loom Cotton Yoga Mat",
      "price": "\u20b91,999.00"
    },
    {
      "name": "Pose Cork Yoga Mat",
      "price": "\u20b94,799.00"
    }
  ]
}


### Scraping Multiple Websites

In [10]:
import json
from scrapegraphai.graphs import SmartScraperMultiGraph

multiple_search_graph = SmartScraperMultiGraph(
    prompt="Fetch the prices of Llama Models",
    source= [
        "https://www.together.ai/pricing",
        "https://deepinfra.com/"
        ],
    schema=None,
    config=graph_config_ollama
)

result = multiple_search_graph.run()
print(json.dumps(result, indent=4))

--- Executing GraphIterator Node with batchsize 16 ---
processing graph instances:   0%|          | 0/2 [00:00<?, ?it/s]--- Executing Fetch Node ---
--- Executing Fetch Node ---
--- (Fetching HTML from: https://deepinfra.com/) ---
--- (Fetching HTML from: https://www.together.ai/pricing) ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
--- Executing ParseNode Node ---
--- Executing GenerateAnswer Node ---
{}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨
processing graph instances:  50%|█████     | 1/2 [00:55<00:55, 55.29s/it]{'type': 'text', 'data': "I've scraped the following content from a website in Markdown format: \n"}
✨ Try enhanced version of ScrapegraphAI at ]8;;https://scrapegraphai.com\https://scrapegraphai.com]8;;\ ✨
processing graph instances: 100%|██████████| 2/2 [01:46<00:00, 53.05s/it]
--- Executing MergeAnswers Node ---
{'Llama Model Prices': [{'Model': 'Llama-1', 'Price': '$50

{
    "Llama Model Prices": [
        {
            "Model": "Llama-1",
            "Price": "$500-$1000"
        },
        {
            "Model": "Llama-2",
            "Price": "$2000-$3000"
        },
        {
            "Model": "Llama-3",
            "Price": "$800-$1200"
        }
    ],
    "sources": [
        "https://www.together.ai/pricing",
        "https://deepinfra.com/"
    ]
}
